# Визуализация рынка и решений RL-агента

Ноутбук показывает три уровня: синтетические цены и кризисные режимы, итоговый dashboard портфеля и пошаговый replay решений. Он работает с фиксированной политикой сразу после установки проекта; в последнем разделе показано подключение обученной PPO/SAC-модели.

## 1. Подключение Google Drive

В Google Colab проект ожидается в `MyDrive/gas_portfolio_rl`. Ячейка монтирует Drive и проверяет структуру проекта. При локальном запуске автоматически используется текущий каталог проекта.

In [ ]:
from pathlib import Path
import sys

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = Path('/content/drive/MyDrive/gas_portfolio_rl')
else:
    cwd = Path.cwd().resolve()
    if (cwd / 'src' / 'gas_portfolio_rl').exists():
        PROJECT_ROOT = cwd
    elif (cwd / 'gas_portfolio_rl' / 'src' / 'gas_portfolio_rl').exists():
        PROJECT_ROOT = cwd / 'gas_portfolio_rl'
    else:
        raise RuntimeError('Откройте notebook из корня RL_project или gas_portfolio_rl')

sys.path.insert(0, str(PROJECT_ROOT))
from notebook_bootstrap import bootstrap_project
PROJECT_ROOT = bootstrap_project(PROJECT_ROOT, install_dependencies=False)

print('Runtime:', 'Google Colab' if IN_COLAB else 'local')
print('Project root:', PROJECT_ROOT)

## 2. Установка окружения

В Colab базовая установка включает Gymnasium, NumPy, Matplotlib и PyYAML. Поставьте `INSTALL_RL_DEPENDENCIES = True`, если в этом runtime нужно загружать или обучать PPO/SAC через Stable-Baselines3. При локальном запуске notebook использует исходники напрямую и не переустанавливает пакет.

In [ ]:
INSTALL_RL_DEPENDENCIES = False
PROJECT_ROOT = bootstrap_project(
    PROJECT_ROOT,
    install_dependencies=IN_COLAB,
    install_rl_dependencies=INSTALL_RL_DEPENDENCIES,
)

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output, display

from gas_portfolio_rl import AppConfig, GasPortfolioEnv
from gas_portfolio_rl.simulation import MarketSimulator
from gas_portfolio_rl.visualization import plot_forward_curve, plot_market_prices, plot_portfolio_dashboard

REPORTS = PROJECT_ROOT / 'results' / 'reports'
REPORTS.mkdir(parents=True, exist_ok=True)
SEED = 42
print('Package imported from:', __import__('gas_portfolio_rl').__file__)
print('Environment is ready. Reports:', REPORTS)

## 3. Что измеряет среда

**Наблюдение агента:** циклические признаки даты оценки и даты поставки; последние доходности газа, GasCo и фьючерса; rolling volatility GasCo и фьючерса; rolling correlation; spot/front basis; наклоны и кривизна forward curve; winter-summer spread; сроки до экспирации и следующего дивиденда; текущие позиции; cash; margin ratio; drawdown. Истинный режим рынка и параметры сезонного профиля скрыты.

**Действие:** `[stock_weight, futures_exposure]`. Доля акции лежит в `[0, 1]`, номинальная фьючерсная экспозиция — в `[-1, 1]`. Фьючерсная exposure не является долей вложенного капитала: фьючерс имеет нулевую стоимость актива, но создаёт variation P&L и требование к обеспечению.

**Диагностика:** `info` и `env.trajectory` содержат цены, cash, стоимость портфеля, P&L, leverage, позиции, drawdown, margin, turnover, издержки и скрытый режим — только для анализа, не для policy.

In [ ]:
config = AppConfig.from_yaml(
    PROJECT_ROOT / 'configs' / 'market' / 'default.yaml',
    PROJECT_ROOT / 'configs' / 'environment' / 'default.yaml',
)
print('Episode steps:', config.market.episode_steps)
print('Initial capital:', f'{config.portfolio.initial_cash:,.0f}')
print('Margin rate:', f'{config.portfolio.margin_rate:.0%}')

## 4. Отдельная визуализация цен

Красная заливка показывает истинные кризисные периоды. Она используется только при анализе симулятора и не попадает в observation агента. Второй график показывает доступную агенту 12-контрактную forward curve с ежемесячными датами поставки 27-го числа.

In [ ]:
states = MarketSimulator(config.market).simulate(config.market.episode_steps, seed=SEED)
prices_path = REPORTS / 'notebook_market_prices.png'
price_figure, _ = plot_market_prices(states, prices_path)
plt.show()
forward_path = REPORTS / 'notebook_forward_curve.png'
forward_figure, _ = plot_forward_curve(states[0], forward_path)
plt.show()
print('Saved:', prices_path)
print('Saved:', forward_path)

## 5. Запуск политики и сбор trajectory

По умолчанию используется понятная фиксированная стратегия: 55% GasCo и короткая фьючерсная exposure −25%. Для случайной политики замените `POLICY_MODE` на `random`.

In [ ]:
POLICY_MODE = 'fixed'  # 'fixed' или 'random'
FIXED_ACTION = np.asarray([0.55, -0.25], dtype=np.float32)

env = GasPortfolioEnv(config, render_mode='ansi')
observation, info = env.reset(seed=SEED)
terminated = truncated = False
episode_reward = 0.0

while not (terminated or truncated):
    action = env.action_space.sample() if POLICY_MODE == 'random' else FIXED_ACTION
    observation, reward, terminated, truncated, info = env.step(action)
    episode_reward += reward

print(env.render())
print('Episode reward:', round(episode_reward, 6))
print('Margin calls:', info['margin_calls'])
print('Cumulative costs:', f"{info['cumulative_transaction_cost']:,.2f}")

## 6. Dashboard состояния портфеля

Leverage считается как `stock_weight + abs(futures_exposure)`. На панели allocation отрицательная линия фьючерса означает short hedge.

In [ ]:
dashboard_path = REPORTS / 'notebook_portfolio_dashboard.png'
dashboard_figure, _ = plot_portfolio_dashboard(env.trajectory, dashboard_path)
plt.show()
print('Saved:', dashboard_path)

## 7. Пошаговый replay

Поставьте `RUN_REPLAY = True`, чтобы проиграть уже собранную траекторию. `FRAME_STEP` управляет частотой обновления: 10 означает один кадр на 10 торговых дней.

In [ ]:
RUN_REPLAY = False
FRAME_STEP = 10

if RUN_REPLAY:
    for end in range(2, len(env.trajectory) + 1, FRAME_STEP):
        clear_output(wait=True)
        figure, _ = plot_portfolio_dashboard(env.trajectory[:end])
        display(figure)
        plt.close(figure)
    clear_output(wait=True)
    final_figure, _ = plot_portfolio_dashboard(env.trajectory)
    display(final_figure)
    plt.close(final_figure)
else:
    print('Replay выключен. Установите RUN_REPLAY = True и перезапустите ячейку.')

## 8. Подключение обученной PPO/SAC-модели

Визуализация не зависит от алгоритма. Любая Stable-Baselines3 модель должна только вернуть действие через `predict`.

In [ ]:
def run_trained_model(model, config, seed=42):
    model_env = GasPortfolioEnv(config)
    observation, info = model_env.reset(seed=seed)
    terminated = truncated = False
    while not (terminated or truncated):
        action, _ = model.predict(observation, deterministic=True)
        observation, reward, terminated, truncated, info = model_env.step(action)
    return model_env, info

# Пример после обучения:
# from stable_baselines3 import PPO
# model = PPO.load(PROJECT_ROOT / 'results' / 'runs' / '<run>' / 'best_model.zip')
# trained_env, trained_info = run_trained_model(model, config, seed=SEED)
# figure, _ = plot_portfolio_dashboard(trained_env.trajectory)
# plt.show()

print('Функция run_trained_model готова.')